# Interactive Agent Session: Chapter2_LangGraph_Support

This Jupyter notebook provides a sandbox to run AI Agent code securely block-by-block. Make sure you have your `.env` configured properly before executing the agent loop below.

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI

# 1. Define the internal memory passed between graph nodes
class AgentState(TypedDict):
    ticket_text: str
    intent: str
    response: str
    actions: list

llm = ChatOpenAI(model="gpt-4-turbo")

# 2. Define standard Python functions as 'Nodes' in the graph
def classify_intent_node(state: AgentState):
    prompt = f"Determine the intent of this ticket: '{state['ticket_text']}'. Output exactly 'refund' or 'technical'."
    result = llm.invoke(prompt).content.strip().lower()
    return {"intent": result, "actions": ["classified"]}

def refund_agent_node(state: AgentState):
    # E.g. requests.post('api.stripe.com/refund')
    return {"response": "Refund processed.", "actions": state["actions"] + ["refunded"]}

def technical_agent_node(state: AgentState):
    result = llm.invoke(f"Draft a polite troubleshooting email for: {state['ticket_text']}")
    return {"response": result.content, "actions": state["actions"] + ["drafted_email"]}

# 3. Routing Edge Function
def route_ticket(state: AgentState) -> Literal["refund_agent", "technical_agent"]:
    if "refund" in state["intent"]: return "refund_agent"
    return "technical_agent"

# 4. Compile the Graph Architecture
workflow = StateGraph(AgentState)
workflow.add_node("classifier", classify_intent_node)
workflow.add_node("refund_agent", refund_agent_node)
workflow.add_node("technical_agent", technical_agent_node)

workflow.set_entry_point("classifier")
workflow.add_conditional_edges("classifier", route_ticket)

workflow.add_edge("refund_agent", END)
workflow.add_edge("technical_agent", END)
app = workflow.compile()

# 5. Execute
final_state = app.invoke({"ticket_text": "I want my money back.", "actions": []})
print(final_state["response"])
